# Smart Farming AI — Phase 1 free-GPU pipeline
Drive stores archives/checkpoints; Colab local disk performs training.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Smart_Farming_AI')
for p in ['datasets/raw','datasets/processed','datasets/field_test','models/checkpoints','models/onnx','results','notebooks']:
    (ROOT/p).mkdir(parents=True, exist_ok=True)
print(ROOT)

In [ ]:
# Put one crop ZIP in datasets/raw before running. Work locally for fast I/O.
CROP = 'rice'
ZIP = ROOT/'datasets/raw'/f'{CROP}.zip'
LOCAL = Path('/content/dataset')
assert ZIP.exists(), f'Missing {ZIP}'
!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -q "{ZIP}" -d /content/dataset


In [ ]:
# Clone/update code, install dependencies, then audit before training.
!git clone -q https://github.com/mdrafiullah1830/smart_farming_ai.git /content/smart_farming_ai || true
%cd /content/smart_farming_ai
!pip -q install torch torchvision scikit-learn pandas pillow onnx onnxruntime
!python scripts/audit_disease_dataset.py /content/dataset --output "{ROOT}/datasets/processed/{CROP}_manifest.csv"
!python ai_models/disease_detection/train_pytorch_onnx.py --data-dir /content/dataset --output-dir "{ROOT}/models/checkpoints" --epochs 20
